In [9]:
import pandas as pd

In [8]:
pd.read_csv("Adult_Census_Income.csv").convert_dtypes().info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             32561 non-null  Int64 
 1   workclass       32561 non-null  string
 2   fnlwgt          32561 non-null  Int64 
 3   education       32561 non-null  string
 4   education.num   32561 non-null  Int64 
 5   marital.status  32561 non-null  string
 6   occupation      32561 non-null  string
 7   relationship    32561 non-null  string
 8   race            32561 non-null  string
 9   sex             32561 non-null  string
 10  capital.gain    32561 non-null  Int64 
 11  capital.loss    32561 non-null  Int64 
 12  hours.per.week  32561 non-null  Int64 
 13  native.country  32561 non-null  string
 14  income          32561 non-null  string
dtypes: Int64(6), string(9)
memory usage: 3.9 MB


In [6]:
import pandas as pd
import json

def enrich_feature_meta(csv_path: str, json_path: str, output_path: str):
    # 读取 CSV 数据
    df = pd.read_csv(csv_path)
    
    # 替换 DataFrame 列名中的 '.' 为 '_'
    df.columns = [col.replace('.', '_') for col in df.columns]

    # 读取 JSON 文件并将 key 中的 '-' 替换为 '_'
    with open(json_path, 'r') as f:
        raw_meta = json.load(f)

    # 替换 JSON 的 key
    original_meta = {k.replace('-', '_'): v for k, v in raw_meta.items()}

    # 构建 enriched metadata
    enriched_meta = {}
    for col in df.columns:
        desc = original_meta.get(col, "")  # 若 JSON 中无该列，desc设为空字符串
        col_data = df[col]

        meta = {
            "description": desc
        }

        # 判断类型
        if col_data.nunique() == len(col_data):  # 每个值都不同
            meta["type"] = "unique_identifier"
            meta["unique_values"] = int(col_data.nunique())
        elif col_data.dtype == object or col_data.dtype.name == "category":
            meta["type"] = "categorical"
            meta["values"] = sorted(col_data.dropna().unique().tolist())
        else:
            meta["type"] = "numerical"

        enriched_meta[col] = meta

    # 保存为新 JSON 文件
    with open(output_path, 'w') as f:
        json.dump(enriched_meta, f, indent=2)

    print(f"✅ 已保存 enriched metadata 至：{output_path}")


In [7]:
enrich_feature_meta(
    csv_path="Adult_Census_Income.csv",
    json_path="adult-metadata.json",
    output_path="parsed_description.json"
)


✅ 已保存 enriched metadata 至：parsed_description.json
